In [1]:
!pip install numpy pandas tensorflow scikit-learn keras

  Using cached tensorflow-2.21.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (4.4 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.81.1-cp312-cp312-macosx_11_0_universal2.whl.metadata (3.7 kB)
  Using cached h5py-3.14.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.5.4-cp312-cp312-macosx_10_13_universal2.whl.metadata (8.9 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached optree-0.19.1-cp312

In [2]:
import numpy as np
import pandas as pd
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
reviews = [
    "The movie was excellent and engaging.",
    "A boring and dull story.",
    "Truly an amazing performance.",
    "The worst acting I have ever seen.",
    "Loved every minute of this film.",
    "Completely predictable and uninspired.",
    "A masterpiece of modern cinema.",
    "I regret spending my money on this.",
    "Fantastic plot with great actors.",
    "So bad, I walked out halfway through."
]

# Corresponding sentiment labels (1 for positive, 0 for negative)
sentiments = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

# Create a DataFrame
df = pd.DataFrame({'review': reviews, 'sentiment': sentiments})
print("Sample Data:")
print(df)

Sample Data:
                                   review  sentiment
0   The movie was excellent and engaging.          1
1                A boring and dull story.          0
2           Truly an amazing performance.          1
3      The worst acting I have ever seen.          0
4        Loved every minute of this film.          1
5  Completely predictable and uninspired.          0
6         A masterpiece of modern cinema.          1
7     I regret spending my money on this.          0
8       Fantastic plot with great actors.          1
9   So bad, I walked out halfway through.          0


In [4]:
# Use Tokenizer to convert text to sequences of integers
max_words = 100 # Consider the top 100 most frequent words
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['review'])
X = tokenizer.texts_to_matrix(df['review'], mode='count')

# Encode the sentiment labels
y = np.array(df['sentiment'])

print("\nShape of input data (X):", X.shape)
print("Vocabulary size:", len(tokenizer.word_index))
print("\nFirst review vectorized:")
print(X[0])



Shape of input data (X): (10, 100)
Vocabulary size: 48

First review vectorized:
[0. 1. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


In [5]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])



Training samples: 8
Testing samples: 2


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

# Build the feedforward neural network
model = Sequential()

# Explicit input layer
model.add(Input(shape=(max_words,)))

# Hidden layer with ReLU activation
model.add(Dense(64, activation='relu'))
# Output layer with sigmoid activation for binary classification
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         6,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,529 (25.50 KB)

 Trainable params: 6,529 (25.50 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:

# Train the model
# 'epochs' is the number of times the entire dataset is passed forward and backward through the network
history = model.fit(X_train, y_train, epochs=10, batch_size=2, validation_split=0.2, verbose=1)



Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 1.0000 - loss: 0.6066 - val_accuracy: 0.5000 - val_loss: 0.6634
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.5835 - val_accuracy: 0.5000 - val_loss: 0.6640
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.5618 - val_accuracy: 0.5000 - val_loss: 0.6637
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.5424 - val_accuracy: 0.5000 - val_loss: 0.6637
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.5233 - val_accuracy: 0.5000 - val_loss: 0.6636
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.5044 - val_accuracy: 0.5000 - val_loss: 0.6633
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.4867 - val_accuracy: 0.5000 - val_loss: 0.6629
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.4700 - val_accuracy: 0.5000 - val_loss: 0.6626
Epoch 9

In [8]:
# Evaluate the model on the test data
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {accuracy*100:.2f}%")

# Make a prediction on new, unseen text
new_review = "The special effects were fantastic, but the plot was weak."
new_review_vec = tokenizer.texts_to_matrix([new_review], mode='count')

# Predict the sentiment
prediction = model.predict(new_review_vec)
sentiment_label = "Positive" if prediction[0][0] > 0.5 else "Negative"

print(f"\nNew review: '{new_review}'")
print(f"Prediction probability: {prediction[0][0]:.4f}")
print(f"Predicted sentiment: {sentiment_label}")



Test Accuracy: 100.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step

New review: 'The special effects were fantastic, but the plot was weak.'
Prediction probability: 0.5857
Predicted sentiment: Positive
